# 05 · Compare saved plant classifiers with Meta Muse Spark

This Colab notebook evaluates **Muse Spark 1.3** against the EfficientNet-B2, DINOv3, and SigLIP2 runs saved by notebook 03. It reads their **test predictions** from Google Drive and scores every system on the **same image IDs** from the new internal test split. It does not retrain or load those models.

The default pilot selects one test image per class (**18 API requests**). Set `IMAGES_PER_CLASS = None` to evaluate all **111 test images**. Meta Model API calls may incur charges. Each successful response is cached to Drive immediately.

Muse Spark receives the image and the same 18 class names available to the trained heads. This is a closed-set plant identity comparison, not a pet-toxicity assessment. `UNKNOWN` and invalid answers count as incorrect and are reported as abstentions. Small pilots have high uncertainty.

References: [Meta Model API image understanding](https://dev.meta.ai/docs/image-understanding), [models](https://dev.meta.ai/docs/models), [pricing](https://dev.meta.ai/docs/pricing-rate-limits).

## 1. Install the API client and choose the evaluation size

The dataset ZIP and trained-run artifacts live in Google Drive. Add `MODEL_API_KEY` to Colab Secrets and grant notebook access. The key is read only when an uncached API call is needed and is never saved. Images are read directly from the packaged ZIP; no Oxford download or duplicate image archive is created.

In [ ]:
import subprocess
import sys
subprocess.check_call([sys.executable, '-m', 'pip', '-q', 'install', 'openai>=1.0,<3'])

In [ ]:
from pathlib import Path
from zipfile import ZipFile
import base64
import hashlib
import json
import os
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import f1_score
from openai import OpenAI

SEED = 42
DATASET_VERSION = 'v20261026_2355546'
IMAGES_PER_CLASS = 1  # 1 = 18-call pilot; None = full fixed test set (111 calls)
MODEL_ID = 'muse-spark-1.3'
SAVE_TO_DRIVE = True
assert IMAGES_PER_CLASS is None or IMAGES_PER_CLASS >= 1

IN_COLAB = 'google.colab' in sys.modules
if SAVE_TO_DRIVE:
    if not IN_COLAB:
        raise RuntimeError('Set SAVE_TO_DRIVE=False for a local run, or use Google Colab.')
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT = Path('/content/drive/MyDrive')
    BASE_RUNS_DIR = DRIVE_ROOT / 'AI/meta-global-hackaton/runs'
    ARCHIVE_PATH = DRIVE_ROOT / 'AI/meta-global-hackaton/data' / f'plant_export_{DATASET_VERSION}_holdout10_seed{SEED}.zip'
else:
    BASE_RUNS_DIR = Path('./data/processed/runs').resolve()
    ARCHIVE_PATH = Path('./data/processed') / f'plant_export_{DATASET_VERSION}_holdout10_seed{SEED}.zip'
EXPERIMENT_ID = f'plant_export_{DATASET_VERSION}_holdout10_seed{SEED}'
RUNS_DIR = BASE_RUNS_DIR / EXPERIMENT_ID
if not RUNS_DIR.is_dir():
    raise FileNotFoundError(f'No saved runs at {RUNS_DIR}. Complete notebook 03 first.')
if not ARCHIVE_PATH.is_file():
    raise FileNotFoundError(f'Dataset archive not found: {ARCHIVE_PATH}')
print('Reading run artifacts from:', RUNS_DIR)

## 2. Load the fixed test manifest and saved predictions

Run directories must agree on the dataset fingerprint and class mapping. Their prediction CSVs must exactly cover the saved test manifest. For a pilot, the same number of examples is sampled per class using seed 42. Setting `IMAGES_PER_CLASS = None` selects all 111 new test images.

In [ ]:
def read_json(path: Path) -> dict:
    return json.loads(path.read_text(encoding='utf-8'))


test_path = RUNS_DIR / 'test.csv'
if not test_path.is_file():
    raise FileNotFoundError(f'Missing {test_path}; rerun notebook 03 to export the test manifest.')
test_df = pd.read_csv(test_path).sort_values('image_id').reset_index(drop=True)
assert test_df.image_id.is_unique and test_df.target.nunique() == 18
if IMAGES_PER_CLASS is None:
    sample = test_df.copy()
else:
    assert test_df.groupby('target').size().min() >= IMAGES_PER_CLASS, (
        'IMAGES_PER_CLASS exceeds the smallest per-class test count.'
    )
    sample = (test_df.groupby('target', group_keys=False)
              .sample(n=IMAGES_PER_CLASS, random_state=SEED)
              .sort_values('image_id').reset_index(drop=True))

runs = {}
reference_fingerprint = None
reference_mapping = None
for name in ('efficientnet_b2', 'dinov3', 'siglip2'):
    folder = RUNS_DIR / name
    required = [folder / part for part in
                ('dataset_metadata.json', 'class_mapping.json', 'test_predictions.csv')]
    if not all(path.is_file() for path in required):
        print('Skipping incomplete run:', name)
        continue
    metadata = read_json(required[0])
    mapping = read_json(required[1])
    fingerprint = metadata['dataset_fingerprint']
    if metadata['version_tag'] != DATASET_VERSION:
        raise ValueError(f'{name} is not from {DATASET_VERSION}.')
    if reference_fingerprint is None:
        reference_fingerprint = fingerprint
        reference_mapping = mapping['index_to_class']
    if fingerprint != reference_fingerprint or mapping['index_to_class'] != reference_mapping:
        raise ValueError(f'{name} uses a different dataset or class mapping.')
    predictions = pd.read_csv(required[2])
    if predictions.image_id.duplicated().any() or set(predictions.image_id) != set(test_df.image_id):
        raise ValueError(f'{name} predictions do not exactly match test.csv.')
    aligned = sample[['image_id', 'target']].merge(
        predictions[['image_id', 'true_index', 'predicted_index']],
        on='image_id', validate='one_to_one'
    )
    if not (aligned.target == aligned.true_index).all():
        raise ValueError(f'{name} true labels disagree with test.csv.')
    runs[name] = aligned
if not runs:
    raise FileNotFoundError('No complete training run with test predictions was found.')
CLASS_NAMES = reference_mapping
if len(CLASS_NAMES) != 18 or set(sample.target) != set(range(18)):
    raise ValueError('Expected the same 18 class indices from notebook 03.')
print('Comparable trained runs:', list(runs))
print('Selected test images:', len(sample), '| Dataset fingerprint:', reference_fingerprint)
display(sample[['image_id', 'class_name', 'source_split']].head())

## 3. Verify test photographs against the Drive manifest

The images are read directly from the packaged ZIP. Every selected image is SHA-256 checked against the manifest written by notebook 03 before it can be sent to Meta. A missing member or mismatched hash stops the notebook.

In [ ]:
with ZipFile(ARCHIVE_PATH) as archive:
    for row in sample.itertuples(index=False):
        try:
            image_bytes = archive.read(row.zip_member)
        except KeyError as exc:
            raise FileNotFoundError(f'Missing ZIP member: {row.zip_member}') from exc
        if hashlib.sha256(image_bytes).hexdigest() != row.sha256:
            raise ValueError(f'Image hash mismatch for image_id={row.image_id}')
print('Verified all', len(sample), 'selected image hashes.')

## 4. Ask Muse Spark to identify each plant

The prompt lists the same 18 possible names. The model may answer `UNKNOWN` when the image is too ambiguous. Only an exact class name is accepted as a class prediction. Other text is saved as an invalid answer and scored as a miss. Each response is appended to Drive with prompt/completion token usage and an estimated cost using Meta's published rates for the selected model. Cached input tokens are charged at the lower cached-input rate when usage reports them. Older cached responses remain usable, but their missing usage is reported as unknown. The estimate is not a billing record.

Rates: [Meta Model API pricing](https://dev.meta.ai/docs/pricing-rate-limits), checked 2026-10-27.

In [ ]:
CLASS_LIST = '\n'.join(f'- {name}' for name in CLASS_NAMES)
PROMPT = (
    'Identify the plant in this photograph. Choose exactly one name from this list, '
    'or answer UNKNOWN if the image is too ambiguous to identify reliably. '
    'Return only the class name or UNKNOWN, with no explanation.\n' + CLASS_LIST
)
PROMPT_HASH = hashlib.sha256(PROMPT.encode()).hexdigest()[:12]
CACHE_PATH = RUNS_DIR / f'muse_spark_{MODEL_ID.replace("-", "_")}_{PROMPT_HASH}_{reference_fingerprint}.csv'
USAGE_COLUMNS = ['input_tokens', 'cached_input_tokens', 'output_tokens', 'estimated_cost_usd']
CACHE_COLUMNS = ['image_id', 'model_id', 'prompt_hash', 'dataset_fingerprint',
                 'raw_answer', 'predicted_index', 'status', *USAGE_COLUMNS]
MODEL_PRICING_PER_MILLION = {
    'muse-spark-1.3': {'input': 1.25, 'cached_input': 0.15, 'output': 4.25},
    'muse-spark-1.2': {'input': 1.25, 'cached_input': 0.15, 'output': 4.25},
    'muse-spark-1.1': {'input': 1.25, 'cached_input': 0.15, 'output': 4.25},
    'muse-spark-1.3-contributor': {'input': 0.10, 'cached_input': 0.002, 'output': 0.20},
    'muse-spark-1.2-contributor': {'input': 0.10, 'cached_input': 0.002, 'output': 0.20},
}
if MODEL_ID not in MODEL_PRICING_PER_MILLION:
    raise ValueError(f'No official pricing configured for {MODEL_ID}. Update MODEL_PRICING_PER_MILLION.')
MODEL_PRICING = MODEL_PRICING_PER_MILLION[MODEL_ID]

if CACHE_PATH.is_file():
    cache = pd.read_csv(CACHE_PATH).fillna({'raw_answer': '', 'status': ''})
    required_columns = set(CACHE_COLUMNS[:7])
    if required_columns - set(cache.columns):
        raise ValueError(f'Cache schema is incomplete: {CACHE_PATH}')
    # Older response caches predate token accounting. Keep their predictions,
    # but mark their usage as unknown instead of reporting a misleading zero.
    migrated = False
    for column in USAGE_COLUMNS:
        if column not in cache:
            cache[column] = np.nan
            migrated = True
    # Earlier usage caches priced all prompt tokens at the standard input rate.
    # Treat their already-recorded cost as uncached rather than changing history.
    if 'cached_input_tokens' not in pd.read_csv(CACHE_PATH, nrows=0).columns and \
            {'input_tokens', 'estimated_cost_usd'} <= set(cache.columns):
        known_usage = cache.input_tokens.notna() & cache.estimated_cost_usd.notna()
        cache.loc[known_usage, 'cached_input_tokens'] = 0
    cache = cache[CACHE_COLUMNS]
    if migrated:
        cache.to_csv(CACHE_PATH, index=False)
    if cache.image_id.duplicated().any():
        raise ValueError(f'Duplicate image IDs in cache: {CACHE_PATH}')
    if not ((cache.model_id == MODEL_ID) & (cache.prompt_hash == PROMPT_HASH) &
            (cache.dataset_fingerprint == reference_fingerprint)).all():
        raise ValueError(f'Cache metadata does not match this evaluation: {CACHE_PATH}')
else:
    cache = pd.DataFrame(columns=CACHE_COLUMNS)

name_to_index = {name.casefold(): index for index, name in enumerate(CLASS_NAMES)}


def parse_answer(answer: str) -> tuple[int, str]:
    normalized = answer.strip().strip('"\'').strip().casefold()
    if normalized == 'unknown':
        return -1, 'abstained'
    if normalized in name_to_index:
        return name_to_index[normalized], 'answered'
    return -1, 'invalid'


pending = sample[~sample.image_id.isin(cache.image_id)]
print('Cached:', len(sample) - len(pending), '| API calls pending:', len(pending))
new_api_calls = len(pending)
if len(pending):
    try:
        from google.colab import userdata
        api_key = userdata.get('MODEL_API_KEY')
    except (ImportError, KeyError):
        api_key = os.environ.get('MODEL_API_KEY')
    if not api_key:
        raise RuntimeError('Set MODEL_API_KEY in Colab Secrets or the environment.')
    client = OpenAI(base_url='https://api.meta.ai/v1', api_key=api_key,
                    timeout=90.0, max_retries=2)
    del api_key
    with ZipFile(ARCHIVE_PATH) as archive:
        for row in pending.itertuples(index=False):
            image_bytes = archive.read(row.zip_member)
            mime_type = {'.jpg': 'image/jpeg', '.jpeg': 'image/jpeg',
                         '.png': 'image/png', '.webp': 'image/webp'}[Path(row.zip_member).suffix.lower()]
            image_url = f'data:{mime_type};base64,' + base64.b64encode(image_bytes).decode('ascii')
            response = client.chat.completions.create(
                model=MODEL_ID,
                messages=[{'role': 'user', 'content': [
                    {'type': 'text', 'text': PROMPT},
                    {'type': 'image_url', 'image_url': {'url': image_url}},
                ]}],
                max_completion_tokens=512,
                reasoning_effort='minimal',
                seed=SEED,
            )
            answer = response.choices[0].message.content or ''
            predicted_index, status = parse_answer(answer)
            usage = response.usage
            input_tokens = int(usage.prompt_tokens) if usage and usage.prompt_tokens is not None else np.nan
            prompt_details = getattr(usage, 'prompt_tokens_details', None) if usage else None
            cached_input_tokens = (
                int(getattr(prompt_details, 'cached_tokens', 0) or 0)
                if usage and pd.notna(input_tokens) else np.nan
            )
            output_tokens = int(usage.completion_tokens) if usage and usage.completion_tokens is not None else np.nan
            uncached_input_tokens = max(input_tokens - cached_input_tokens, 0) if pd.notna(input_tokens) and pd.notna(cached_input_tokens) else np.nan
            estimated_cost = (
                uncached_input_tokens * MODEL_PRICING['input'] / 1_000_000
                + cached_input_tokens * MODEL_PRICING['cached_input'] / 1_000_000
                + output_tokens * MODEL_PRICING['output'] / 1_000_000
            ) if pd.notna(uncached_input_tokens) and pd.notna(output_tokens) else np.nan

            print(f'Input tokens: {input_tokens}')
            print(f'Cached input tokens: {cached_input_tokens}')
            print(f'Output tokens: {output_tokens}')
            if pd.notna(estimated_cost):
                print(f'Estimated cost: ${estimated_cost:.6f}')
            else:
                print('Estimated cost: unavailable (usage was not returned)')

            cache.loc[len(cache)] = [row.image_id, MODEL_ID, PROMPT_HASH,
                                     reference_fingerprint, answer, predicted_index, status,
                                     input_tokens, cached_input_tokens, output_tokens, estimated_cost]
            cache.to_csv(CACHE_PATH, index=False)
            print(f'image {row.image_id}: {status} ({len(cache)}/{len(sample)} cached)')

spark = sample[['image_id', 'target']].merge(
    cache[['image_id', 'predicted_index', 'status']], on='image_id',
    how='left', validate='one_to_one'
)
if spark.predicted_index.isna().any():
    raise RuntimeError('Some selected images have no Muse Spark response.')
spark['predicted_index'] = spark.predicted_index.astype(int)
print('Saved API responses:', CACHE_PATH)
display(spark.status.value_counts().rename_axis('status').to_frame('count'))

sample_usage = cache[cache.image_id.isin(sample.image_id)].copy()
for column in USAGE_COLUMNS:
    sample_usage[column] = pd.to_numeric(sample_usage[column], errors='coerce')
usage_known = sample_usage[USAGE_COLUMNS].notna().all(axis=1)
usage_report = pd.DataFrame([{
    'responses_for_selected_images': len(sample_usage),
    'responses_with_usage': int(usage_known.sum()),
    'new_api_requests_this_execution': new_api_calls,
    'input_tokens': sample_usage.loc[usage_known, 'input_tokens'].sum(min_count=1),
    'cached_input_tokens': sample_usage.loc[usage_known, 'cached_input_tokens'].sum(min_count=1),
    'output_tokens': sample_usage.loc[usage_known, 'output_tokens'].sum(min_count=1),
    'estimated_cost_usd': sample_usage.loc[usage_known, 'estimated_cost_usd'].sum(min_count=1),
}])
print('Token and estimated cost totals (older cached responses may have unknown usage):')
display(usage_report)


## 5. Compare all models on identical image IDs

Accuracy counts `UNKNOWN` and invalid API responses as errors; coverage shows how often Muse Spark returned one of the 18 names. The default one-per-class pilot is too small for stable rankings. The baselines were selected with validation data in notebook 03; this test sample is for comparison only.

In [ ]:
comparison = []
per_class_rows = []
for name, frame in [*runs.items(), (MODEL_ID, spark)]:
    actual = frame.target.to_numpy(dtype=int)
    predicted = frame.predicted_index.to_numpy(dtype=int)
    answered = predicted >= 0
    comparison.append({
        'model': name,
        'images': len(frame),
        'accuracy': float(np.mean(actual == predicted)),
        'macro_f1': float(f1_score(actual, predicted, labels=list(range(len(CLASS_NAMES))),
                                   average='macro', zero_division=0)),
        'coverage': float(np.mean(answered)),
        'accuracy_when_answered': float(np.mean(actual[answered] == predicted[answered]))
                                  if answered.any() else np.nan,
    })
    for class_index, class_name in enumerate(CLASS_NAMES):
        mask = actual == class_index
        per_class_rows.append({'model': name, 'class_name': class_name,
                               'correct': int((actual[mask] == predicted[mask]).sum()),
                               'total': int(mask.sum())})
results = pd.DataFrame(comparison).sort_values('accuracy', ascending=False)
display(results.style.format({'accuracy': '{:.3f}', 'macro_f1': '{:.3f}',
                              'coverage': '{:.3f}', 'accuracy_when_answered': '{:.3f}'}))

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(results.model, results.accuracy, color=['#355c7d', '#6c5b7b', '#c06c84', '#f8b195'][:len(results)])
ax.set_ylim(0, 1)
ax.set_ylabel('Accuracy on identical selected test images')
ax.set_title(f'Plant identity · {len(sample)} fixed-test images')
ax.tick_params(axis='x', labelrotation=20)
fig.tight_layout()
plt.show()

per_class = pd.DataFrame(per_class_rows)
per_class['accuracy'] = per_class.correct / per_class.total
pivot = per_class.pivot(index='class_name', columns='model', values='accuracy').reindex(CLASS_NAMES)
fig, ax = plt.subplots(figsize=(8, 9))
heat = ax.imshow(pivot, vmin=0, vmax=1, cmap='viridis', aspect='auto')
ax.set_xticks(range(len(pivot.columns)), pivot.columns, rotation=20, ha='right')
ax.set_yticks(range(len(pivot.index)), pivot.index)
ax.set_title('Per-class correctness on the same sampled images')
fig.colorbar(heat, ax=ax, label='Fraction correct')
fig.tight_layout()
plt.show()

## Interpretation and next steps

- Fill in observations **after** running the notebook: which plant classes show disagreements? Inspect those images and raw Spark answers in the Drive cache.
- Repeat with `IMAGES_PER_CLASS = None` for all 111 internal test images. Some classes still have only three test examples, so report class support and uncertainty.
- Spark sees the class list in its prompt, while the trained heads learned those names as supervised targets. Their training and inference costs differ; include those conditions when reporting numbers.
- The archive's class names do not verify the species in an individual photograph or whether a real pet exposure is safe. Keep identification uncertainty separate from any later verified toxicity workflow.